# Hunyuan3D 2.0 — Генерация 3D моделей

Запусти этот notebook в Google Colab с GPU (Runtime → Change runtime type → T4 GPU).

**Что делает:** загружает картинку → генерирует 3D модель → скачивает GLB файл.

**Время:** ~2-5 минут на генерацию.

## 1. Проверка GPU

In [ ]:
!nvidia-smi

## 2. Установка зависимостей

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!git clone https://github.com/Tencent/Hunyuan3D-2.git
%cd Hunyuan3D-2
!pip install -r requirements.txt
!pip install -e .

## 3. Код генерации

Загрузи свою картинку (справа → Files → Upload) или используй демо.

In [ ]:
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.texgen import Hunyuan3DPaintPipeline
import shutil
from pathlib import Path

# Загружаем модель геометрии
print("Загрузка Hunyuan3D-DiT...")
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
print("Геометрия загружена!")

# Загружаем модель текстур
print("Загрузка Hunyuan3D-Paint...")
texture_pipeline = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
print("Текстуры загружены!")

## 4. Загрузка картинки

Вариант A: загрузи файл через Files → Upload

Вариант B: вставь URL картинки

In [ ]:
import urllib.request

# Вариант A: загруженный файл
# Замени путь на свой файл
INPUT_IMAGE = "/content/grogu.png"  # ← замени на свой путь

# Вариант B: скачать по URL (раскомментируй)
#!wget -O /content/grogu.png "URL_КАРТИНКИ"
#INPUT_IMAGE = "/content/grogu.png"

print(f"Используем: {INPUT_IMAGE}")

## 5. Генерация 3D модели

In [ ]:
print("Генерация геометрии...")
mesh = shape_pipeline(image=INPUT_IMAGE)[0]

print("Генерация текстур...")
textured_mesh = texture_pipeline(mesh, image=INPUT_IMAGE)

# Сохраняем
output_path = "/content/grogu_hunyuan3d.glb"
textured_mesh.export(output_path)
print(f"Сохранено: {output_path}")

## 6. Скачивание результата

Нажми на ссылку чтобы скачать GLB файл.

In [ ]:
from google.colab import files
import os

size = os.path.getsize(output_path)
print(f"Размер: {size:,} bytes ({size/1024/1024:.1f} MB)")

files.download(output_path)

## 7. (Опционально) Конвертация в STL

In [ ]:
!pip install trimesh

import trimesh

scene = trimesh.load(output_path)
if isinstance(scene, trimesh.Scene):
    meshes = list(scene.geometry.values())
    combined = trimesh.util.concatenate(meshes)
else:
    combined = scene

# Масштаб до 70мм высоты
h = combined.bounding_box.extents[2]
combined.apply_scale(70.0 / h)

stl_path = "/content/grogu_hunyuan3d.stl"
combined.export(stl_path)
print(f"STL: {os.path.getsize(stl_path):,} bytes")

files.download(stl_path)